# Statistics and experimentation

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/concepts/statistics.ipynb)

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"numpy": "numpy>=1.26", "pandas": "pandas>=2.0", "scipy": "scipy>=1.11"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## Descriptive statistics

### Why averages lie and percentiles do not

The mean is the balance point of the histogram, so one heavy value drags it; the median splits the count in half and barely moves. Latency objectives are therefore written on p95 and p99. A p50 of 14 ms with a p99 of 900 ms means one request in a hundred is unusable even though the mean looks fine.

- **Nearest-rank** percentiles return a value that really occurred; **interpolated** ones (SQL `percentile_cont`, numpy's default) can return a value no user saw.
- Percentiles are **not additive**: the daily p95 is not the average of hourly p95s. Pool raw events or keep a sketch (t-digest) per window.
- With fewer than 100 samples, "p99" is just the maximum.
- A bimodal mix of 5 ms and 500 ms has a mean of 252 ms that no request is anywhere near.

In [ ]:
import math
import statistics


def nearest_rank(ordered: list[float], pct: float) -> float:
    """Smallest observation at or above the pct-th rank."""
    rank = max(1, math.ceil(pct / 100 * len(ordered)))
    return ordered[rank - 1]


def percentile_cont(ordered: list[float], p: float) -> float:
    """Linear interpolation between the closest ranks, as SQL percentile_cont does."""
    position = (len(ordered) - 1) * p
    low, high = math.floor(position), math.ceil(position)
    if low == high:
        return ordered[low]
    return ordered[low] * (high - position) + ordered[high] * (position - low)


latencies = sorted([10, 12, 13, 15, 20, 900])
mean = statistics.fmean(latencies)
p50_rank, p50_interp = nearest_rank(latencies, 50), percentile_cont(latencies, 0.5)
print(f"mean={mean:.1f} p50(nearest)={p50_rank} p50(interpolated)={p50_interp} p99={nearest_rank(latencies, 99)}")
assert round(mean, 1) == 161.7
assert p50_rank == 13 and p50_interp == 14.0  # the interpolated median never occurred
assert nearest_rank(latencies, 99) == 900

### Center, spread and shape

These are three separate questions: mean and median answer center, standard deviation and IQR answer spread, skew answers shape. `mean > median` signals a right tail.

- Sample standard deviation divides by `n - 1` (SQL `stddev_samp`, numpy `ddof=1`); population divides by `n` (`stddev_pop`, numpy's default `ddof=0`). Mixing them changes answers on small groups.
- The **coefficient of variation** (`stdev / mean`) compares spread across metrics with different scales; it is meaningless when the mean is near zero.

In [ ]:
import numpy as np
from scipy import stats

values = [2, 4, 4, 4, 5, 5, 7, 9]
sample_sd, population_sd = statistics.stdev(values), statistics.pstdev(values)
cv = sample_sd / statistics.fmean(values)
print(f"mean=5.0 sample_sd={sample_sd:.2f} population_sd={population_sd:.2f} cv={cv:.2f}")
assert round(sample_sd, 2) == 2.14 and population_sd == 2.0 and round(cv, 2) == 0.43
assert np.isclose(np.std(values, ddof=1), sample_sd) and np.isclose(np.std(values), population_sd)

skewed = [10, 12, 13, 15, 20, 900]
print(f"skew of latency sample = {stats.skew(skewed):.2f}")
assert stats.skew(skewed) > 1  # strong right tail

### Heavy tails and the log scale

Latency, revenue per user and file sizes are right-skewed, often roughly **log-normal** (the log of the value is normal). For a log-normal, `mean = exp(mu + sigma^2/2)` and `median = exp(mu)`, so the tail weight alone drives the mean-to-median ratio. The 68/95/99.7 rule holds on the log scale, not the raw scale.

A heavy tail inflates the standard deviation, which hides the very outliers a 3-sigma rule is meant to catch and makes t tests underpowered. Aggregate on quantiles or logs, use Mann-Whitney U or the bootstrap for tests, and never average ratios (revenue per session): sum numerators and denominators separately.

In [ ]:
rng = np.random.default_rng(7)
lognormal = rng.lognormal(mean=3.0, sigma=1.2, size=20_000)
ratio = lognormal.mean() / np.median(lognormal)
logged = np.log(lognormal)
within_2sd_log = np.mean(np.abs(logged - logged.mean()) < 2 * logged.std())
print(f"mean/median = {ratio:.2f} (theory {math.exp(1.2**2 / 2):.2f}); share within 2 sd on log scale = {within_2sd_log:.3f}")
assert 1.8 < ratio < 2.3
assert 0.94 < within_2sd_log < 0.96  # the normal rule works after the log transform

## Sampling without bias

A sample is representative when every unit has a known, non-zero chance of selection that does not correlate with the metric. **Simple random** sampling is the default; **stratified** sampling guarantees coverage of small segments; **cluster** sampling draws whole groups (files, users); **systematic** (every k-th row) fails when the data has a period matching k. **Hash sampling** (`hash(id) % 100 < 5`) is consistent across jobs and backfills.

Common bugs: taking the first N rows of a time-ordered table (biased toward old records), and filtering after sampling (changes the denominator). Filter the population first, sample second, and record the seed or salt.

In [ ]:
import hashlib
import random
from collections import Counter, defaultdict

population = [{"user_id": i, "segment": "enterprise" if i >= 95_000 else "self_serve"} for i in range(100_000)]

first_n = population[:10_000]  # ids were created in cohort order, enterprises onboarded last


def stratified(rows: list[dict], key: str, per_stratum: int, seed: int = 7) -> list[dict]:
    buckets: dict[str, list[dict]] = defaultdict(list)
    for row in rows:
        buckets[row[key]].append(row)
    rng = random.Random(seed)
    return [row for bucket in buckets.values() for row in rng.sample(bucket, min(per_stratum, len(bucket)))]


def in_hash_sample(user_id: int, pct: int = 5, salt: str = "s1") -> bool:
    return int(hashlib.sha256(f"{salt}:{user_id}".encode()).hexdigest()[:8], 16) % 100 < pct


strat = stratified(population, "segment", 2_000)
hashed = [row for row in population if in_hash_sample(row["user_id"])]
print("first 10k:", Counter(r["segment"] for r in first_n))
print("stratified:", Counter(r["segment"] for r in strat))
print(f"hash sample size: {len(hashed)} (~5%)")
assert Counter(r["segment"] for r in first_n)["enterprise"] == 0
assert Counter(r["segment"] for r in strat) == {"self_serve": 2_000, "enterprise": 2_000}
assert 4_500 < len(hashed) < 5_500
assert hashed == [row for row in population if in_hash_sample(row["user_id"])]  # reproducible

## Inference

### Confidence intervals

The interval is `estimate +/- z * SE` with `SE = sd / sqrt(n)`, so halving the width needs four times the data. A 95% CI is a **procedure** that covers the true parameter in 95% of repeated samples; it is *not* "a 95% probability the parameter is in this interval" (that is a Bayesian credible interval and needs a prior).

- Use a t multiplier for small n, and Wilson or Clopper-Pearson for proportions when `n * p < 5`.
- Many rows per user inflate effective variance: compute the interval on per-user aggregates.
- Overlapping intervals are not a significance test, and a biased sample gives a tight interval around the wrong number.

The simulation below checks the coverage claim directly.

In [ ]:
def ci_mean(sample: np.ndarray, z: float = 1.959964) -> tuple[float, float]:
    se = sample.std(ddof=1) / math.sqrt(sample.size)
    return sample.mean() - z * se, sample.mean() + z * se


rng = np.random.default_rng(0)
true_mean = 20.0
covered = 0
for _ in range(2_000):
    low, high = ci_mean(rng.normal(true_mean, 10.0, size=100))
    covered += low <= true_mean <= high
coverage = covered / 2_000
print(f"coverage of nominal 95% intervals: {coverage:.3f}")
assert 0.93 < coverage < 0.97

se = 10 / math.sqrt(100)
print(f"n=100, mean=20, sd=10 -> 95% CI [{20 - 1.959964 * se:.2f}, {20 + 1.959964 * se:.2f}]")
assert round(20 - 1.959964 * se, 2) == 18.04

### t test and two-proportion z test

Both divide an observed difference by its standard error. **Welch's t test** (no equal-variance assumption) is the safe default for means; the **two-proportion z test** pools the rates under the null. Report the effect size and interval with the p-value: a non-significant result from an underpowered test is fully consistent with a real lift.

Assumptions to check: independent observations (one row per randomization unit), at least ~5 successes and failures per arm, and a direction fixed before looking if you go one-sided. Use Mann-Whitney U for heavy-tailed metrics.

In [ ]:
def two_proportion_z(success_a: int, n_a: int, success_b: int, n_b: int) -> dict[str, float]:
    p_a, p_b = success_a / n_a, success_b / n_b
    pooled = (success_a + success_b) / (n_a + n_b)
    se = math.sqrt(pooled * (1 - pooled) * (1 / n_a + 1 / n_b))
    z = (p_b - p_a) / se
    return {"z": z, "p_value": math.erfc(abs(z) / math.sqrt(2)), "lift": p_b - p_a}


result = two_proportion_z(10, 100, 18, 100)
print(f"10/100 vs 18/100: z={result['z']:.3f} p={result['p_value']:.3f} lift={result['lift']:.2f}")
assert round(result["z"], 3) == 1.630 and round(result["p_value"], 3) == 0.103

rng = np.random.default_rng(1)
control = rng.normal(20.0, 5.0, 500)
treatment = rng.normal(21.5, 8.0, 500)
welch = stats.ttest_ind(control, treatment, equal_var=False)
mwu = stats.mannwhitneyu(control, treatment)
print(f"Welch t={welch.statistic:.2f} p={welch.pvalue:.2e}; Mann-Whitney p={mwu.pvalue:.2e}")
assert welch.pvalue < 0.05

### Chi-square test on categorical data

Chi-square compares observed counts with those expected under independence (`row_total * col_total / grand_total`), with `(rows - 1) * (cols - 1)` degrees of freedom. It tests any association, not a direction. Because big tables make trivial associations significant, report **Cramer's V** as the effect size, inspect standardized residuals to see which cell drove it, and merge categories or use Fisher's exact test when an expected count is below 5. Use counts, never percentages.

In [ ]:
table = [[200, 800], [250, 750]]
chi2, p_value, dof, expected = stats.chi2_contingency(table, correction=False)
n = sum(map(sum, table))
cramers_v = math.sqrt(chi2 / (n * (min(len(table), len(table[0])) - 1)))
print(f"chi2={chi2:.2f} dof={dof} p={p_value:.4f} V={cramers_v:.3f} expected={expected.tolist()}")
assert round(chi2, 2) == 7.17 and round(p_value, 4) == 0.0074
assert expected[0][0] == 225  # statistically clear, practically tiny (V = 0.06)

# The same statistic by hand: one degree of freedom gives p = erfc(sqrt(chi2 / 2)).
rows, cols = [sum(r) for r in table], [sum(c) for c in zip(*table)]
by_hand = sum((table[i][j] - rows[i] * cols[j] / n) ** 2 / (rows[i] * cols[j] / n) for i in range(2) for j in range(2))
assert math.isclose(by_hand, chi2) and math.isclose(math.erfc(math.sqrt(by_hand / 2)), p_value)

### What p-values mean and multiple comparisons

A p-value is `P(data at least this extreme | null)`. It is not the probability the null is true, nor the probability of replication. Under the null it is uniform on [0, 1], so at alpha 0.05 twenty A/A tests produce about one false positive by construction.

- **Bonferroni** (`alpha / m`) controls the chance of *any* false positive; conservative.
- **Benjamini-Hochberg** rejects the largest rank k with `p_(k) <= k/m * alpha`, controlling the expected *share* of false discoveries; the better default for metric dashboards.
- With enough data everything is significant: always pair a p-value with an effect size and interval.

In [ ]:
def bonferroni(p_values: list[float], alpha: float = 0.05) -> list[int]:
    return [i for i, p in enumerate(p_values) if p <= alpha / len(p_values)]


def benjamini_hochberg(p_values: list[float], alpha: float = 0.05) -> list[int]:
    m = len(p_values)
    ranked = sorted(enumerate(p_values), key=lambda item: item[1])
    cutoff = max((rank for rank, (_, p) in enumerate(ranked, start=1) if p <= rank / m * alpha), default=0)
    return sorted(index for index, _ in ranked[:cutoff])


p_values = [0.001, 0.004, 0.01, 0.02, 0.03, 0.2, 0.5, 0.7, 0.9, 0.95]
print("Bonferroni rejects:", bonferroni(p_values), "BH rejects:", benjamini_hochberg(p_values))
assert len(bonferroni(p_values)) == 2 and len(benjamini_hochberg(p_values)) == 4

# Twenty A/A tests: p-values are uniform, so ~1 crosses 0.05 by chance.
rng = np.random.default_rng(3)
aa_p = [stats.ttest_ind(rng.normal(size=200), rng.normal(size=200)).pvalue for _ in range(2_000)]
false_positive_rate = np.mean(np.array(aa_p) < 0.05)
print(f"A/A false-positive rate: {false_positive_rate:.3f}")
assert 0.04 < false_positive_rate < 0.06

### Statistical power and sample size

Power (1 - beta) is the probability of detecting an effect of a stated size when it exists. Sample size grows with variance and with the power demanded, and falls with the **square** of the effect: halving the minimum detectable effect quadruples n. A rule of thumb for proportions is `16 * p(1-p) / delta^2` per arm at alpha 0.05 and 80% power.

Never compute "observed power" after a non-significant result: it is a monotone function of the p-value and adds nothing. The formula assumes independent Bernoulli units, so it understates n for ratio metrics or clustered data.

In [ ]:
def sample_size_per_arm(p1: float, p2: float, alpha: float = 0.05, power: float = 0.8) -> int:
    z_alpha, z_power = stats.norm.ppf(1 - alpha / 2), stats.norm.ppf(power)
    pooled = (p1 + p2) / 2
    numerator = (z_alpha * math.sqrt(2 * pooled * (1 - pooled)) + z_power * math.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2
    return math.ceil(numerator / (p2 - p1) ** 2)


n_2pt, n_1pt = sample_size_per_arm(0.10, 0.12), sample_size_per_arm(0.10, 0.11)
print(f"10% -> 12%: {n_2pt:,} per arm; 10% -> 11%: {n_1pt:,} per arm")
assert n_2pt == 3841 and n_1pt == 14751
assert 3.5 < n_1pt / n_2pt < 4.0  # half the effect, ~4x the sample

## Experimentation

### Designing an A/B test

Randomization makes the arms exchangeable on average, so the outcome difference estimates the effect, provided assignment is **random, stable and independent of the metric**. Hash the randomization unit with a fixed salt, keep it for the whole experiment, and analyze at that unit (one row per user, not per session).

Pre-register the primary metric, guardrails, unit, minimum detectable effect and analysis window. Typical breakages: randomizing by session but analyzing by user, changing the salt mid-flight, users leaking between arms (network effects), and ramp-ups that expose heavy users first. For platform changes (file format, shuffle setting, compaction policy) randomize at a unit that cannot leak, such as a tenant or cluster; a before/after comparison is a time series, not an experiment.

In [ ]:
def bucket(unit_id: str, experiment: str, salt: str = "v1", buckets: int = 100) -> int:
    digest = hashlib.sha256(f"{experiment}:{salt}:{unit_id}".encode()).hexdigest()
    return int(digest[:8], 16) % buckets


def variant(unit_id: str, experiment: str, treatment_share: float = 0.5) -> str:
    return "treatment" if bucket(unit_id, experiment) < treatment_share * 100 else "control"


users = [f"user_{i}" for i in range(20_000)]
split = Counter(variant(u, "checkout_v2") for u in users)
print(split)
assert all(variant(u, "checkout_v2") == variant(u, "checkout_v2") for u in users[:100])  # stable
assert abs(split["treatment"] / len(users) - 0.5) < 0.02
# A different experiment name re-randomizes independently.
overlap = sum(variant(u, "checkout_v2") == variant(u, "search_v3") for u in users) / len(users)
assert abs(overlap - 0.5) < 0.02

### Sample ratio mismatch (SRM)

SRM means the observed split differs from the designed split by more than chance allows. It is a data-integrity alarm: assignment or logging is broken (bots filtered on one arm, a redirect dropping events), and the arms are no longer comparable. Run a chi-square goodness-of-fit test overall and per platform, country and device, with a strict alpha such as 0.001 because it runs constantly. Never "correct" SRM by reweighting: find the bug and rerun.

In [ ]:
def srm_check(observed: list[int], ratios: list[float], alpha: float = 0.001) -> dict[str, float]:
    total = sum(observed)
    statistic, p_value = stats.chisquare(observed, [r * total for r in ratios])
    return {"chi2": float(statistic), "p_value": float(p_value), "srm": bool(p_value < alpha)}


bad = srm_check([49_100, 50_900], [0.5, 0.5])
ok = srm_check([49_900, 50_100], [0.5, 0.5])
print(f"49,100/50,900: chi2={bad['chi2']:.1f} p={bad['p_value']:.1e} srm={bad['srm']}; 49,900/50,100 srm={ok['srm']}")
assert round(bad["chi2"], 1) == 32.4 and bad["srm"] and not ok["srm"]

### Guardrails and novelty effects

The primary metric answers "did it work"; guardrails (latency, error rate, refunds, unsubscribes, data freshness) answer "did it break something". Pre-register each guardrail with a direction and a non-inferiority threshold. A +1.5% conversion lift with a +40% p99 latency regression is not a win.

A **novelty effect** is an early lift that decays as users get used to the change; judge on the mature window, and remember a seasonal week can mimic the decay.

In [ ]:
def guardrail_breached(control: list[float], treatment: list[float], max_relative_increase: float) -> bool:
    relative = statistics.fmean(treatment) / statistics.fmean(control) - 1
    return relative > max_relative_increase


p99_control = [300, 310, 295, 305]
p99_treatment = [420, 430, 415, 425]
assert guardrail_breached(p99_control, p99_treatment, max_relative_increase=0.05)

daily_lift = [0.060, 0.045, 0.032, 0.022, 0.016, 0.013, 0.012, 0.012, 0.011, 0.012, 0.011, 0.012, 0.012, 0.011]
whole_run, mature = statistics.fmean(daily_lift), statistics.fmean(daily_lift[7:])
print(f"whole-run lift {whole_run:.3f} vs mature-window lift {mature:.3f}")
assert whole_run > 1.5 * mature  # novelty inflates the naive estimate

### Sequential testing and peeking

A fixed-horizon alpha assumes one look. Checking daily and stopping at the first p < 0.05 gives many chances to cross, so the real false-positive rate climbs well above 5%. Fix the look schedule in advance and use an alpha-spending boundary (Pocock, O'Brien-Fleming), or an always-valid confidence sequence that needs no schedule. The simulation is an A/A test, so every rejection is a false positive.

In [ ]:
def peeking_false_positive_rate(z_threshold: float, looks: int = 10, per_look: int = 500,
                                trials: int = 2_000, baseline: float = 0.1, seed: int = 0) -> float:
    rng = np.random.default_rng(seed)
    n = np.arange(1, looks + 1) * per_look
    a = rng.binomial(per_look, baseline, size=(trials, looks)).cumsum(axis=1)
    b = rng.binomial(per_look, baseline, size=(trials, looks)).cumsum(axis=1)
    pooled = (a + b) / (2 * n)
    z = (b - a) / n / np.sqrt(pooled * (1 - pooled) * 2 / n)
    return float(np.mean((np.abs(z) > z_threshold).any(axis=1)))


naive = peeking_false_positive_rate(z_threshold=1.96)
pocock = peeking_false_positive_rate(z_threshold=2.56)  # Pocock constant boundary for 10 looks, alpha 0.05
print(f"10 naive looks: {naive:.3f}; Pocock boundary: {pocock:.3f}")
assert naive > 0.15
assert 0.03 < pocock < 0.07

### Bayesian vs frequentist

Frequentists fix the parameter and treat the data as random; Bayesians fix the data and put a distribution on the parameter. With a Beta prior and binomial data the posterior is `Beta(a + successes, b + failures)`, and `P(B > A)` is the share of posterior draws where B wins, which maps directly to a ship decision. Pair it with the **expected loss** of choosing B. A flat prior is still a choice and moves the answer at low traffic; do not call a credible interval a confidence interval.

In [ ]:
rng = np.random.default_rng(0)
draws = 200_000
post_a = rng.beta(1 + 100, 1 + 900, draws)
post_b = rng.beta(1 + 130, 1 + 870, draws)
prob_b_better = float(np.mean(post_b > post_a))
expected_loss_b = float(np.mean(np.maximum(post_a - post_b, 0)))
print(f"P(B > A) = {prob_b_better:.3f}, expected loss of shipping B = {expected_loss_b:.5f}")
assert prob_b_better > 0.97
assert expected_loss_b < 0.001

## Causality and paradoxes

### Correlation, confounding and causation

A **confounder** is a common cause of treatment and outcome (temperature drives ice cream sales and drownings; customer size drives revenue and support tickets). Regression or partial correlation removes the *linear* part of a *measured* confounder; unmeasured ones still bias the estimate. Adjusting for a **mediator** removes a real effect and adjusting for a **collider** creates a fake one, so choose the adjustment set from the causal graph. Clean causal evidence needs a randomized experiment or a credible quasi-experiment (difference-in-differences, geo holdouts).

In [ ]:
rng = np.random.default_rng(11)
temperature = rng.normal(25, 5, 2_000)
ice_cream = 3 * temperature + rng.normal(0, 5, 2_000)
drownings = 0.5 * temperature + rng.normal(0, 2, 2_000)


def residualize(y: np.ndarray, z: np.ndarray) -> np.ndarray:
    design = np.column_stack([np.ones_like(z), z])
    return y - design @ np.linalg.lstsq(design, y, rcond=None)[0]


raw_r = np.corrcoef(ice_cream, drownings)[0, 1]
partial_r = np.corrcoef(residualize(ice_cream, temperature), residualize(drownings, temperature))[0, 1]
print(f"raw correlation {raw_r:.2f}, partial correlation given temperature {partial_r:.2f}")
assert raw_r > 0.5 and abs(partial_r) < 0.05

### Simpson's paradox

A pooled comparison weights each stratum by its traffic. When exposure is uneven across strata, that weighting can reverse the sign of an effect that holds within every stratum. Report the stratified estimate with explicit weights from the target population. A shifting traffic mix over time produces the same false trend.

In [ ]:
import pandas as pd

groups = pd.DataFrame(
    {
        "segment": ["mobile", "desktop"],
        "control_n": [100, 900], "control_conv": [10, 270],
        "treatment_n": [900, 100], "treatment_conv": [117, 32],
    }
)
groups["lift"] = groups.treatment_conv / groups.treatment_n - groups.control_conv / groups.control_n
pooled_lift = groups.treatment_conv.sum() / groups.treatment_n.sum() - groups.control_conv.sum() / groups.control_n.sum()
weights = (groups.control_n + groups.treatment_n) / (groups.control_n + groups.treatment_n).sum()
stratified_lift = float((weights * groups.lift).sum())
print(groups[["segment", "lift"]].round(3).to_string(index=False))
print(f"pooled lift {pooled_lift:+.3f} vs stratified lift {stratified_lift:+.3f}")
assert (groups.lift > 0).all() and pooled_lift < 0 < stratified_lift

## Anomaly detection and time series

### z-score, IQR and MAD

All three ask "how far is this point from normal", with different definitions of normal. The **z-score** uses mean and standard deviation, both dragged by the outlier itself. The **IQR rule** flags points beyond `Q1 - 1.5 IQR` or `Q3 + 1.5 IQR`. The **robust z-score** `0.6745 * (x - median) / MAD` is the right default for heavy-tailed metrics.

For alerting, learn the baseline over a rolling window with robust statistics, require a minimum business-relevant magnitude as well as a statistical flag, compare against the same season, and measure the alert's precision and recall on past incidents. A fixed "row count within 10%" rule is either noisy or blind when variance changes with weekday or volume.

In [ ]:
def zscore(values: list[float], x: float) -> float:
    return abs(x - statistics.fmean(values)) / statistics.stdev(values)


def robust_z(values: list[float], x: float) -> float:
    median = statistics.median(values)
    mad = statistics.median([abs(v - median) for v in values]) or 1e-9
    return 0.6745 * abs(x - median) / mad


def iqr_bounds(values: list[float], k: float = 1.5) -> tuple[float, float]:
    q1, _, q3 = statistics.quantiles(values, n=4)
    return q1 - k * (q3 - q1), q3 + k * (q3 - q1)


values = [10, 11, 12, 12, 13, 900]
print(f"z={zscore(values, 900):.2f} robust_z={robust_z(values, 900):.1f} iqr_bounds={iqr_bounds(values)}")
assert zscore(values, 900) < 3.0  # the plain z-score misses it: 900 inflated the stdev
assert robust_z(values, 900) > 3.5 and 900 > iqr_bounds(values)[1]

history = [100, 101, 99, 102, 98, 103, 100, 101, 100, 620]
median = statistics.median(history)
upper = median + 3.5 * statistics.median([abs(v - median) for v in history]) / 0.6745
print(f"alert threshold from history with a past spike: {upper:.1f}")
assert round(upper, 1) == 105.7 and 140 > upper > 104

### Trend, seasonality and stationarity

Decomposition splits a series into trend, seasonal and residual parts: additive when seasonal amplitude is constant, multiplicative when it grows with the level (a log makes it additive). A centered moving average estimates the trend, the mean detrended value per phase gives the seasonal index, and the residual is where anomalies live. Production tools (STL) also handle changing seasonality; holidays belong in an event calendar, not the seasonal index.

A **stationary** series has constant mean and variance. Trending series are not, and correlating two of them gives spurious regression. First differencing removes a linear trend, a seasonal difference removes a repeating pattern, and a log stabilizes growing variance. Formal tests: ADF (null: unit root) and KPSS (null: stationary).

In [ ]:
series = [10, 12, 15, 11, 13, 16, 12, 14, 17]
period = 3
trend = pd.Series(series).rolling(period, center=True).mean()
detrended = pd.Series(series) - trend
seasonal = detrended.groupby(np.arange(len(series)) % period).mean()
seasonal -= seasonal.mean()
print("trend:", trend.round(2).tolist())
print("seasonal index by phase:", seasonal.round(2).tolist())
assert trend.dropna().is_monotonic_increasing
assert seasonal.idxmax() == 2 and seasonal.idxmin() == 0

rng = np.random.default_rng(5)
a = np.cumsum(rng.normal(size=500))  # two independent random walks
b = np.cumsum(rng.normal(size=500))
levels_r, diffs_r = np.corrcoef(a, b)[0, 1], np.corrcoef(np.diff(a), np.diff(b))[0, 1]
print(f"random walks: correlation of levels {levels_r:.2f}, of differences {diffs_r:.2f}")
assert abs(diffs_r) < 0.1  # differencing removes the spurious relationship

## Bootstrap confidence intervals

The bootstrap resamples the observed data with replacement to approximate the sampling distribution of any statistic, which is the practical answer for medians, ratios and p99s. Resample the **cluster** (user) rather than rows when users contribute many rows, keep the seed, and use more draws for extreme quantiles (or BCa intervals for small samples).

In [ ]:
values = np.array([10, 12, 13, 15, 20, 900])
rng = np.random.default_rng(0)
medians = np.median(rng.choice(values, size=(5_000, values.size), replace=True), axis=1)
low, high = np.quantile(medians, [0.025, 0.975])
print(f"median={np.median(values)} 95% bootstrap CI=[{low}, {high}]")
assert low <= np.median(values) <= high and high < values.max()

scipy_ci = stats.bootstrap((values,), np.median, n_resamples=5_000, method="percentile", random_state=0).confidence_interval
assert scipy_ci.low <= 14 <= scipy_ci.high

## Capacity planning with percentiles

An SLO is a statement about a percentile over a window ("99% of requests under 300 ms over 28 days"). Plan on a high percentile of demand plus explicit headroom, because queues grow non-linearly as utilization approaches 1 (going from 70% to 90% utilization more than doubles the queue). Little's law links them: `L = lambda * W`.

`replicas = ceil(peak * headroom / usable_per_replica)`, where usable capacity falls once a replica's own p99 exceeds the latency budget. Use the seasonal peak, not the daily one, and leave headroom for autoscaling reaction time. A p99 caused by a stop-the-world pause does not shrink with more replicas.

In [ ]:
def replicas_for_peak(peak_rps: float, per_replica_rps: float, p99_ms: float,
                      latency_budget_ms: float = 300.0, headroom: float = 1.3) -> int:
    usable = per_replica_rps * min(1.0, latency_budget_ms / p99_ms)
    return math.ceil(peak_rps * headroom / usable)


def mm1_queue_length(utilization: float) -> float:
    return utilization**2 / (1 - utilization)


print("replicas, healthy p99:", replicas_for_peak(12_000, 2_000, p99_ms=250))
print("replicas, slow p99:", replicas_for_peak(12_000, 2_000, p99_ms=450))
print(f"M/M/1 queue length at 70% vs 90%: {mm1_queue_length(0.7):.2f} vs {mm1_queue_length(0.9):.2f}")
assert replicas_for_peak(12_000, 2_000, p99_ms=250) == 8
assert replicas_for_peak(12_000, 2_000, p99_ms=450) == 12
assert mm1_queue_length(0.9) > 2 * mm1_queue_length(0.7)

## Statistics in SQL

Warehouses expose the same primitives: `percentile_cont` (interpolated) and `percentile_disc` (an actual value), `stddev_samp` / `stddev_pop`, `approx_percentile` for scale, and `width_bucket` for histograms. Push aggregation into SQL so raw rows never move, compute percentiles at the right grain (they cannot be averaged across partitions), and remember aggregates silently skip NULLs.

```sql
select
  count(*)                                                 as n,
  avg(latency_ms)                                          as mean,
  stddev_samp(latency_ms)                                  as sd,
  percentile_cont(0.50) within group (order by latency_ms) as p50,
  percentile_cont(0.99) within group (order by latency_ms) as p99
from metrics.requests
where event_date between date '2026-09-01' and date '2026-09-30';
```

```sql
-- Rolling 7-day mean per service; percentiles stay grouped aggregates
with daily as (
  select event_date, service, avg(latency_ms) as mean_ms,
         percentile_cont(0.95) within group (order by latency_ms) as p95_ms
  from metrics.requests
  group by event_date, service
)
select event_date, service, p95_ms,
       avg(mean_ms) over (partition by service order by event_date
                          rows between 6 preceding and current row) as mean_7d
from daily;
```

The z-score outlier flag is a cross join against a one-row stats CTE; run it on the standard library's `sqlite3` below.

In [ ]:
import sqlite3

con = sqlite3.connect(":memory:")
con.execute("create table requests (event_id integer, latency_ms real)")
con.executemany("insert into requests values (?, ?)", enumerate([10, 11, 12, 12, 13, 14, 12, 11, 13, 900] * 3))
outliers = con.execute(
    """
    with s as (
      select avg(latency_ms) as mean,
             sqrt(sum((latency_ms - (select avg(latency_ms) from requests)) *
                      (latency_ms - (select avg(latency_ms) from requests))) / (count(*) - 1)) as sd
      from requests
    )
    select event_id, round((latency_ms - s.mean) / s.sd, 2) as z
    from requests cross join s
    where abs((latency_ms - s.mean) / s.sd) > 2
    """
).fetchall()
print(outliers)
assert [event_id for event_id, _ in outliers] == [9, 19, 29]

## Exercises

Each exercise states a task and the expected result; the cell computes and asserts it.

**1. Mean vs median.** For latencies `[2, 3, 3, 4, 4, 5, 6, 7, 120]`, compute mean, median and their ratio; call the sample heavy-tailed when the ratio exceeds 1.5. *Expected:* mean 17.11, median 4, ratio 4.28, heavy-tailed.

In [ ]:
latencies = [2, 3, 3, 4, 4, 5, 6, 7, 120]
ratio = statistics.fmean(latencies) / statistics.median(latencies)
print(f"mean={statistics.fmean(latencies):.2f} median={statistics.median(latencies)} ratio={ratio:.2f}")
assert round(statistics.fmean(latencies), 2) == 17.11 and round(ratio, 2) == 4.28 and ratio > 1.5

**2. Nearest-rank percentiles.** For the integers 1 to 20, compute p50 and p95 (rank = `ceil(pct/100 * n)`). *Expected:* p50 = 10, p95 = 19, meaning 95% of values are at or below 19.

In [ ]:
values = list(range(1, 21))
assert nearest_rank(values, 50) == 10 and nearest_rank(values, 95) == 19
print("p50 =", nearest_rank(values, 50), "p95 =", nearest_rank(values, 95))

**3. Detect SRM by hand.** An intended 50/50 test saw 40,500 vs 41,500 users. Compute chi-square and its p-value with `p = erfc(sqrt(chi2 / 2))`. *Expected:* chi2 about 12.2, p below 0.001, so SRM.

In [ ]:
observed = [40_500, 41_500]
expected_each = sum(observed) / 2
chi2 = sum((o - expected_each) ** 2 / expected_each for o in observed)
p_value = math.erfc(math.sqrt(chi2 / 2))
print(f"chi2={chi2:.1f} p={p_value:.1e}")
assert round(chi2, 1) == 12.2 and p_value < 0.001

**4. Practical vs statistical significance.** With one million users per arm, conversion goes from 10.0% to 10.2%. *Expected:* p far below 0.05, yet the lift is only 0.2 points: decide on the effect size, not the p-value.

In [ ]:
big = two_proportion_z(100_000, 1_000_000, 102_000, 1_000_000)
print(f"z={big['z']:.2f} p={big['p_value']:.1e} lift={big['lift']:.4f}")
assert big["p_value"] < 1e-5 and math.isclose(big["lift"], 0.002)

## Real-world use cases

- **Experimentation platform.** Hash-based assignment, an automated SRM gate on every segment, lift with a confidence interval, and a launch block when any pre-registered guardrail regresses.
- **Data-quality thresholds.** Learn each check's distribution over 90 days and alert outside the 0.5th / 99.5th percentile instead of a hard-coded "within 10%", which adapts to weekday patterns and cuts alert fatigue.
- **Streaming capacity.** Size brokers on peak throughput times headroom (`ceil(peak * 1.4 / per_broker)`) and reprovision before a marketing event.
- **Pipeline anomaly alerts.** Freshness, null rate and schema-drift counts monitored with robust z-scores on a rolling baseline.
- **Marketing incrementality.** Geo holdouts with a difference-in-differences estimate instead of a raw spend-revenue correlation.
- **Fraud thresholds.** Pick the operating point from the precision-recall curve to control expected loss; 99.9% accuracy is meaningless when fraud is 0.1% of traffic.

In [ ]:
history = np.random.default_rng(2).normal(1_000_000, 20_000, 90)
low, high = np.quantile(history, [0.005, 0.995])
assert low < 1_000_000 < high and not (low <= 850_000 <= high)

brokers = math.ceil(450_000 * 1.4 / 100_000)
assert brokers == 7


def diff_in_diff(treat_pre: float, treat_post: float, control_pre: float, control_post: float) -> float:
    return (treat_post - treat_pre) - (control_post - control_pre)


assert diff_in_diff(100, 130, 100, 115) == 15  # half the raw +30 was market-wide growth

rng = np.random.default_rng(4)
labels = rng.random(100_000) < 0.001
scores = np.where(labels, rng.normal(0.8, 0.1, labels.size), rng.normal(0.3, 0.15, labels.size))
accuracy_all_legit = 1 - labels.mean()
threshold = 0.6
flagged = scores > threshold
precision = (flagged & labels).sum() / flagged.sum()
recall = (flagged & labels).sum() / labels.sum()
print(f"accuracy of 'never fraud' = {accuracy_all_legit:.4f}; at threshold {threshold}: precision={precision:.2f} recall={recall:.2f}")
assert accuracy_all_legit > 0.998 and recall > 0.9